# 1193. Monthly Transactions I

[LeetCode problem](https://leetcode.com/problems/monthly-transactions-i/)


# 0. Problem

For each month and country, calculate transaction count, approved count, total amount, and approved amount.


# 1. Setup


In [ ]:
import pandas as pd
transaction_rows=[(121,'US','approved',1000,'2018-12-18'),(122,'US','declined',2000,'2018-12-19'),(123,'US','approved',2000,'2019-01-01'),(124,'DE','approved',2000,'2019-01-07'),(125,'DE','declined',500,'2019-01-08')]
transactions_pd=pd.DataFrame(transaction_rows,columns=['id','country','state','amount','trans_date'])
transactions_pd['trans_date']=pd.to_datetime(transactions_pd['trans_date'])


In [ ]:
# In Colab if needed: !pip -q install pyspark
from pyspark.sql import SparkSession, functions as F, types as T
spark=SparkSession.builder.getOrCreate()
schema=T.StructType([T.StructField('id',T.IntegerType(),False),T.StructField('country',T.StringType(),True),T.StructField('state',T.StringType(),False),T.StructField('amount',T.IntegerType(),False),T.StructField('trans_date',T.StringType(),False)])
transactions_spark=spark.createDataFrame(transaction_rows,schema).withColumn('trans_date',F.to_date('trans_date'))
transactions_spark.createOrReplaceTempView('Transactions')


# 2. SQL Solution


In [ ]:
sql_result=spark.sql("""SELECT DATE_FORMAT(trans_date,'yyyy-MM') AS month,country,COUNT(*) AS trans_count,SUM(CASE WHEN state='approved' THEN 1 ELSE 0 END) AS approved_count,SUM(amount) AS trans_total_amount,SUM(CASE WHEN state='approved' THEN amount ELSE 0 END) AS approved_total_amount FROM Transactions GROUP BY DATE_FORMAT(trans_date,'yyyy-MM'),country""")
sql_result.show(truncate=False)


# 3. pandas Solution


In [ ]:
tx=transactions_pd.assign(month=transactions_pd['trans_date'].dt.strftime('%Y-%m'),approved_flag=(transactions_pd['state']=='approved').astype(int),approved_amount=transactions_pd['amount'].where(transactions_pd['state']=='approved',0))
pandas_result=tx.groupby(['month','country'],dropna=False,as_index=False).agg(trans_count=('id','count'),approved_count=('approved_flag','sum'),trans_total_amount=('amount','sum'),approved_total_amount=('approved_amount','sum'))
pandas_result


# 4. PySpark Solution


In [ ]:
spark_result=(transactions_spark.withColumn('month',F.date_format('trans_date','yyyy-MM')).groupBy('month','country').agg(F.count('*').alias('trans_count'),F.sum(F.when(F.col('state')=='approved',1).otherwise(0)).alias('approved_count'),F.sum('amount').alias('trans_total_amount'),F.sum(F.when(F.col('state')=='approved',F.col('amount')).otherwise(0)).alias('approved_total_amount')))
spark_result.show(truncate=False)


# 5. Pattern Mapping

| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| month | `DATE_FORMAT()` | `.dt.strftime()` | `F.date_format()` |
| conditional count | `SUM(CASE WHEN...)` | flag + `.sum()` | `F.sum(F.when(...))` |
| conditional amount | conditional `SUM` | `.where(...,0)` | `F.when(...).otherwise(0)` |
| multi-column group | `GROUP BY` | `.groupby([...])` | `.groupBy(...)` |


# 6. Muscle-Memory Round


In [ ]:
# MUSCLE MEMORY — SQL
# Use temp view: Transactions


In [ ]:
# MUSCLE MEMORY — PANDAS
# Use DataFrame: transactions_pd


In [ ]:
# MUSCLE MEMORY — PYSPARK
# Use DataFrame: transactions_spark
